# 01 · Card table

Hand labels (`data/labels/cards.csv`) joined to the API `/cards` response by ID (spec §5).
Roles are judgment calls that drive advisory warnings and features; they are not ground truth.
The current labels are **drafts (`v0-draft`)**: review every row, then bump `label_version`.

In [ ]:
import os
import json
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)

DATA_SOURCE = os.environ.get("CRDECK_DATA_SOURCE", "real")  # "real" (default) or "synthetic"
from crdeck.config import Paths
paths = Paths.for_source(DATA_SOURCE)
print(paths.banner())

In [ ]:
from crdeck.cards import load_labels, ROLES
from crdeck.config import CARD_LABELS
labels = load_labels(CARD_LABELS)
print(len(labels), "labeled base cards; label versions:", labels["label_version"].unique().tolist())
labels[["name", "elixir", "card_type", "has_evolution", "has_hero", "is_champion", "is_win_condition", "roles", "targets"]].head(10)

## Rows that need your attention
Notes mark draft labels the drafter was unsure about or cards it did not know.

In [ ]:
labels.loc[labels["notes"] != "", ["name", "card_type", "roles", "targets", "is_win_condition", "notes"]]

## Coverage of roles and forms

In [ ]:
role_counts = pd.Series([r for rs in labels["roles_list"] for r in rs]).value_counts().reindex(sorted(ROLES), fill_value=0)
print(role_counts.to_string())
print()
print("with Evolution:", int(labels.has_evolution.sum()), "| with Hero:", int(labels.has_hero.sum()),
      "| champions:", int(labels.is_champion.sum()), "| win conditions:", int(labels.is_win_condition.sum()))

## Issues found in the source CSV (bootstrap report)

In [ ]:
print((paths.root.parent / "labels" / "cards_label_report.md" if paths.is_synthetic else paths.root / "labels" / "cards_label_report.md").read_text(encoding="utf-8"))

## Join to API `/cards`
Needs `python -m crdeck build` (after the API gate). Lists labeled cards with no API match, API cards with no labels (these become `UNKNOWN_CARD` in battles), and values the API overrode.

In [ ]:
counts_file = paths.interim / "counts.json"
if not counts_file.exists():
    from crdeck.pipeline import NoDataError
    raise NoDataError(f"{counts_file} missing: run the gate (`python -m crdeck gate`) and `python -m crdeck build` first.")
counts = json.loads(counts_file.read_text())
print("\n".join(counts["card_attach_report"]))